In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="ticks", context="paper")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 1.5

files = {
    "Acetate": "Acetate(mM).csv",
    "Propionate": "Propionate(mM).csv",
    "Butyrate": "Butyrate(mM).csv",
    "Total Organic Acids": "Total_OrganicAcids.csv"
}

order = ["Acetate", "Propionate", "Butyrate", "Total Organic Acids"]
plot_labels = ["Acetate", "Propionate", "Butyrate", "Total\nOrganic Acids"]
colors = ["#E31A1C", "#1F78B4", "#33A02C", "#FFD700"]

plot_list = []
summary_stats = []

for label in order:
    df = pd.read_csv(files[label])

    control_rows = df[df['KULFFI'] == 'Control']
    if control_rows.empty:
        raise ValueError(f"Control row not found in {label} dataset.")

    donor_cols = [c for c in df.columns if c.startswith('HS-')]
    vals = control_rows[donor_cols].values[0].astype(float)

    if len(vals) == 0 or np.isnan(vals).any():
        raise ValueError(f"Missing data detected in Control row for {label}.")

    cv = (np.std(vals) / np.mean(vals)) * 100
    summary_stats.append({
        "Component": label,
        "CV": cv
    })

    for v in vals:
        plot_list.append({"Component": label, "Concentration (mM)": v})

df_plot = pd.DataFrame(plot_list)
global_max = df_plot["Concentration (mM)"].max()

fig, ax = plt.subplots(figsize=(10, 8))

sns.boxplot(x='Component', y='Concentration (mM)', data=df_plot, order=order,
            palette=colors, ax=ax, width=0.6, showfliers=False, linewidth=1.5)

for patch in ax.patches:
    patch.set_edgecolor('black')
    patch.set_linewidth(1.5)
for line in ax.lines:
    line.set_color('black')
    line.set_linewidth(1.5)

sns.stripplot(x='Component', y='Concentration (mM)', data=df_plot, order=order,
              color=".1", size=9, alpha=0.5, jitter=0.25, ax=ax)

label_y_pos = global_max * 1.08
for i, stat in enumerate(summary_stats):
    idx = order.index(stat['Component'])
    ax.text(idx, label_y_pos, f"CV: {stat['CV']:.1f}%",
            ha='center', va='bottom', fontsize=18, fontweight='bold')

ax.set_ylim(0, global_max * 1.2)
ax.set_xlim(-0.6, 3.6)

ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.set_xlabel('', fontsize=22)
ax.set_ylabel('Concentration (mM)', fontsize=22, fontweight='bold', labelpad=15)
ax.set_xticklabels(plot_labels, fontsize=20, fontweight='bold')

yticks = ax.get_yticks()
ax.set_yticks(yticks)
ax.set_yticklabels([f"{int(y)}" for y in yticks], fontsize=18, fontweight='normal')

ax.tick_params(axis='y', labelsize=18, width=1.5, length=6)
ax.tick_params(axis='x', width=1.5, length=6)

plt.tight_layout()
plt.savefig('Figure_2b.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.close()